# 👗 FashionMNIST — CNN & DenseNet Classifier

**Objectif** : Comparer deux architectures deep learning sur le dataset FashionMNIST :
- Un **CNN personnalisé** (léger, rapide)
- Un **DenseNet** (connections denses, meilleure réutilisation des features)

---
### 📦 Améliorations apportées
| Amélioration | Description |
|---|---|
| ✅ Data Augmentation | RandomHorizontalFlip, RandomCrop, ColorJitter |
| ✅ Learning Rate Scheduler | CosineAnnealingLR pour une convergence optimale |
| ✅ Early Stopping | Arrêt automatique si pas d'amélioration |
| ✅ Visualisations riches | Courbes, matrice de confusion, exemples d'erreurs |
| ✅ Reproductibilité | Seed fixe sur tous les modules |
| ✅ Rapport de comparaison | Tableau récap CNN vs DenseNet |

## 1. 📚 Imports & Configuration

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split

import torchvision
import torchvision.transforms as transforms
import torchvision.datasets as datasets

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

import time
import copy
import warnings
warnings.filterwarnings('ignore')

# ── Reproductibilité ──────────────────────────────────────────────────────────
SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# ── Device ────────────────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'🖥️  Device : {DEVICE}')
print(f'🔥 PyTorch : {torch.__version__}')

# ── Hyperparamètres ───────────────────────────────────────────────────────────
CONFIG = {
    'batch_size'   : 64,
    'epochs'       : 30,
    'lr'           : 1e-3,
    'weight_decay' : 1e-4,
    'patience'     : 7,       # Early stopping
    'val_split'    : 0.1,
}

CLASS_NAMES = [
    'T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
    'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot'
]
print(f'\n✅ Config chargée — {CONFIG["epochs"]} epochs, batch={CONFIG["batch_size"]}')

## 2. 🗂️ Chargement des données & Data Augmentation

In [ ]:
# ── Transforms ────────────────────────────────────────────────────────────────
# Train : augmentation pour une meilleure généralisation
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomCrop(28, padding=4),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize((0.2860,), (0.3530,))   # stats FashionMNIST
])

# Test : seulement normalisation
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.2860,), (0.3530,))
])

# ── Datasets ──────────────────────────────────────────────────────────────────
full_train = datasets.FashionMNIST('./data', train=True,  download=True, transform=train_transform)
test_set   = datasets.FashionMNIST('./data', train=False, download=True, transform=test_transform)

# Split train / validation
val_size   = int(len(full_train) * CONFIG['val_split'])
train_size = len(full_train) - val_size
train_set, val_set = random_split(full_train, [train_size, val_size],
                                  generator=torch.Generator().manual_seed(SEED))

# ── DataLoaders ───────────────────────────────────────────────────────────────
train_loader = DataLoader(train_set, batch_size=CONFIG['batch_size'], shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_set,   batch_size=CONFIG['batch_size'], shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_set,  batch_size=CONFIG['batch_size'], shuffle=False, num_workers=2, pin_memory=True)

print(f'Train   : {len(train_set):,} images')
print(f'Val     : {len(val_set):,} images')
print(f'Test    : {len(test_set):,} images')

In [ ]:
# ── Visualisation du dataset ──────────────────────────────────────────────────
fig, axes = plt.subplots(3, 10, figsize=(15, 5))
fig.suptitle('FashionMNIST — Exemples par classe', fontsize=14, fontweight='bold')

raw_set = datasets.FashionMNIST('./data', train=True, download=False,
                                 transform=transforms.ToTensor())

# Une image par classe, 3 exemples
shown = {i: 0 for i in range(10)}
for img, label in raw_set:
    row = shown[label]
    if row < 3:
        axes[row, label].imshow(img.squeeze(), cmap='gray')
        axes[row, label].axis('off')
        if row == 0:
            axes[row, label].set_title(CLASS_NAMES[label], fontsize=8, rotation=45, ha='left')
        shown[label] += 1
    if all(v >= 3 for v in shown.values()):
        break

plt.tight_layout()
plt.savefig('dataset_preview.png', dpi=120, bbox_inches='tight')
plt.show()

## 3. 🏗️ Architectures des modèles

### 3.1 CNN Personnalisé
Architecture légère avec BatchNorm et Dropout pour régularisation.

In [ ]:
class CNN(nn.Module):
    """
    CNN personnalisé pour FashionMNIST.
    Architecture : Conv→BN→ReLU→Pool (x3) → FC → Softmax
    """
    def __init__(self, num_classes=10, dropout=0.4):
        super().__init__()

        self.features = nn.Sequential(
            # Bloc 1 : 1→32
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),          # 28→14
            nn.Dropout2d(0.1),

            # Bloc 2 : 32→64
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),          # 14→7
            nn.Dropout2d(0.2),

            # Bloc 3 : 64→128
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((3, 3))  # 7→3
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 3 * 3, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout / 2),
            nn.Linear(128, num_classes)
        )

        # Initialisation des poids (Xavier)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


# Test rapide
cnn = CNN().to(DEVICE)
dummy = torch.randn(2, 1, 28, 28).to(DEVICE)
out   = cnn(dummy)
total_params = sum(p.numel() for p in cnn.parameters() if p.requires_grad)
print(f'✅ CNN — Output shape : {out.shape}')
print(f'   Paramètres entraînables : {total_params:,}')

### 3.2 DenseNet personnalisé
Chaque couche reçoit les features maps de **toutes** les couches précédentes → meilleure réutilisation des gradients.

In [ ]:
class DenseLayer(nn.Module):
    """Couche dense : BN → ReLU → Conv1x1 → BN → ReLU → Conv3x3"""
    def __init__(self, in_channels, growth_rate, bottleneck=4):
        super().__init__()
        inter = bottleneck * growth_rate
        self.block = nn.Sequential(
            nn.BatchNorm2d(in_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels, inter, kernel_size=1, bias=False),
            nn.BatchNorm2d(inter),
            nn.ReLU(inplace=True),
            nn.Conv2d(inter, growth_rate, kernel_size=3, padding=1, bias=False),
            nn.Dropout2d(0.1)
        )

    def forward(self, x):
        return torch.cat([x, self.block(x)], dim=1)   # connexion dense


class DenseBlock(nn.Module):
    """Bloc de N DenseLayers"""
    def __init__(self, in_channels, num_layers, growth_rate):
        super().__init__()
        layers = []
        for i in range(num_layers):
            layers.append(DenseLayer(in_channels + i * growth_rate, growth_rate))
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)


class TransitionLayer(nn.Module):
    """Réduction des channels + spatial pooling entre deux DenseBlocks"""
    def __init__(self, in_channels, compression=0.5):
        super().__init__()
        out_channels = int(in_channels * compression)
        self.block = nn.Sequential(
            nn.BatchNorm2d(in_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),
            nn.AvgPool2d(2, 2)
        )

    def forward(self, x):
        return self.block(x)


class DenseNet(nn.Module):
    """
    DenseNet pour FashionMNIST (28x28 grayscale).
    Architecture : stem → Dense1 → Trans1 → Dense2 → Trans2 → Dense3 → GAP → FC
    """
    def __init__(self, growth_rate=16, layers=(4, 4, 4), num_classes=10, compression=0.5):
        super().__init__()
        channels = 2 * growth_rate   # channels initiaux

        # Stem
        self.stem = nn.Sequential(
            nn.Conv2d(1, channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
            nn.ReLU(inplace=True)
        )

        # Dense blocks + Transitions
        blocks = []
        for i, num_layers in enumerate(layers):
            blocks.append(DenseBlock(channels, num_layers, growth_rate))
            channels += num_layers * growth_rate
            if i < len(layers) - 1:           # pas de transition après le dernier bloc
                blocks.append(TransitionLayer(channels, compression))
                channels = int(channels * compression)

        self.dense_blocks = nn.Sequential(*blocks)

        # Tête de classification
        self.head = nn.Sequential(
            nn.BatchNorm2d(channels),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(channels, num_classes)
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.dense_blocks(x)
        return self.head(x)


# Test rapide
dn = DenseNet().to(DEVICE)
out = dn(dummy)
total_params_dn = sum(p.numel() for p in dn.parameters() if p.requires_grad)
print(f'✅ DenseNet — Output shape : {out.shape}')
print(f'   Paramètres entraînables : {total_params_dn:,}')

## 4. 🔧 Utilitaires : Entraînement & Évaluation

In [ ]:
class EarlyStopping:
    """Arrêt automatique si la val_loss ne s'améliore pas pendant `patience` epochs."""
    def __init__(self, patience=7, min_delta=1e-4):
        self.patience   = patience
        self.min_delta  = min_delta
        self.counter    = 0
        self.best_loss  = None
        self.best_state = None

    def __call__(self, val_loss, model):
        if self.best_loss is None or val_loss < self.best_loss - self.min_delta:
            self.best_loss  = val_loss
            self.best_state = copy.deepcopy(model.state_dict())
            self.counter    = 0
        else:
            self.counter += 1
        return self.counter >= self.patience


def train_one_epoch(model, loader, optimizer, criterion, scaler=None):
    model.train()
    total_loss, correct, total = 0., 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()

        if scaler:                            # Mixed precision (GPU)
            with torch.cuda.amp.autocast():
                logits = model(imgs)
                loss   = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = model(imgs)
            loss   = criterion(logits, labels)
            loss.backward()
            optimizer.step()

        total_loss += loss.item() * imgs.size(0)
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += imgs.size(0)

    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0., 0, 0
    all_preds, all_labels = [], []
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        logits = model(imgs)
        loss   = criterion(logits, labels)
        preds  = logits.argmax(1)

        total_loss += loss.item() * imgs.size(0)
        correct    += (preds == labels).sum().item()
        total      += imgs.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    return total_loss / total, correct / total, all_preds, all_labels


def train_model(model, name, epochs=CONFIG['epochs']):
    """Boucle d'entraînement complète avec scheduler et early stopping."""
    criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
    optimizer = optim.AdamW(model.parameters(),
                            lr=CONFIG['lr'],
                            weight_decay=CONFIG['weight_decay'])
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    stopper   = EarlyStopping(patience=CONFIG['patience'])
    scaler    = torch.cuda.amp.GradScaler() if DEVICE.type == 'cuda' else None

    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [], 'lr': []}
    t0 = time.time()

    for epoch in range(1, epochs + 1):
        tr_loss, tr_acc          = train_one_epoch(model, train_loader, optimizer, criterion, scaler)
        vl_loss, vl_acc, _, _   = evaluate(model, val_loader, criterion)
        scheduler.step()

        history['train_loss'].append(tr_loss)
        history['val_loss'].append(vl_loss)
        history['train_acc'].append(tr_acc)
        history['val_acc'].append(vl_acc)
        history['lr'].append(optimizer.param_groups[0]['lr'])

        if epoch % 5 == 0 or epoch == 1:
            print(f'[{name}] Epoch {epoch:02d}/{epochs} | '
                  f'Loss {tr_loss:.4f}/{vl_loss:.4f} | '
                  f'Acc {tr_acc*100:.2f}%/{vl_acc*100:.2f}% | '
                  f'LR {optimizer.param_groups[0]["lr"]:.2e}')

        if stopper(vl_loss, model):
            print(f'⏹️  Early stopping à l\'epoch {epoch}')
            model.load_state_dict(stopper.best_state)
            break

    elapsed = time.time() - t0
    print(f'\n⏱️  Durée totale : {elapsed:.1f}s ({elapsed/60:.1f} min)')
    return history


print('✅ Utilitaires chargés')

## 5. 🚀 Entraînement

In [ ]:
print('=' * 55)
print('🧠  Entraînement CNN')
print('=' * 55)

cnn_model   = CNN().to(DEVICE)
cnn_history = train_model(cnn_model, 'CNN')

In [ ]:
print('=' * 55)
print('🌲  Entraînement DenseNet')
print('=' * 55)

dn_model   = DenseNet().to(DEVICE)
dn_history = train_model(dn_model, 'DenseNet')

## 6. 📊 Évaluation sur le jeu de test

In [ ]:
criterion = nn.CrossEntropyLoss()

_, cnn_acc, cnn_preds, cnn_labels = evaluate(cnn_model, test_loader, criterion)
_, dn_acc,  dn_preds,  dn_labels  = evaluate(dn_model,  test_loader, criterion)

cnn_params = sum(p.numel() for p in cnn_model.parameters() if p.requires_grad)
dn_params  = sum(p.numel() for p in dn_model.parameters()  if p.requires_grad)

print('\n' + '═' * 50)
print(f'  📊  RÉSULTATS FINAUX (Test set)')
print('═' * 50)
print(f'  CNN      → {cnn_acc*100:.2f}%   ({cnn_params:,} params)')
print(f'  DenseNet → {dn_acc*100:.2f}%   ({dn_params:,} params)')
print('═' * 50)

## 7. 📈 Visualisations

In [ ]:
# ── Courbes Loss & Accuracy ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Courbes d\'entraînement — CNN vs DenseNet', fontsize=14, fontweight='bold')

colors = {'CNN': ('#4C8BF5', '#FF6B6B'), 'DenseNet': ('#00C897', '#FFA500')}

for name, hist, c in [('CNN', cnn_history, colors['CNN']),
                       ('DenseNet', dn_history, colors['DenseNet'])]:
    epochs_ran = range(1, len(hist['train_loss']) + 1)
    axes[0].plot(epochs_ran, hist['train_loss'], '--', color=c[0], alpha=0.7, label=f'{name} Train')
    axes[0].plot(epochs_ran, hist['val_loss'],   '-',  color=c[0], linewidth=2, label=f'{name} Val')
    axes[1].plot(epochs_ran, [a*100 for a in hist['train_acc']], '--', color=c[1], alpha=0.7, label=f'{name} Train')
    axes[1].plot(epochs_ran, [a*100 for a in hist['val_acc']],   '-',  color=c[1], linewidth=2, label=f'{name} Val')

axes[0].set(xlabel='Epoch', ylabel='Loss', title='Loss')
axes[1].set(xlabel='Epoch', ylabel='Accuracy (%)', title='Accuracy')
for ax in axes:
    ax.legend()
    ax.grid(alpha=0.3)
    ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── Matrices de confusion ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle('Matrices de Confusion — Test Set', fontsize=14, fontweight='bold')

for ax, preds, labels, title in [
    (axes[0], cnn_preds, cnn_labels, f'CNN  ({cnn_acc*100:.2f}%)'),
    (axes[1], dn_preds,  dn_labels,  f'DenseNet  ({dn_acc*100:.2f}%)')
]:
    cm = confusion_matrix(labels, preds)
    cm_pct = cm.astype('float') / cm.sum(axis=1, keepdims=True) * 100
    sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                ax=ax, linewidths=0.5, cbar_kws={'label': '%'})
    ax.set(title=title, xlabel='Prédit', ylabel='Réel')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=9)

plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── Exemples d'erreurs (DenseNet) ─────────────────────────────────────────────
dn_model.eval()
errors_imgs, errors_true, errors_pred = [], [], []

with torch.no_grad():
    for imgs, labels in test_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        preds = dn_model(imgs).argmax(1)
        mask  = preds != labels
        errors_imgs.extend(imgs[mask].cpu())
        errors_true.extend(labels[mask].cpu().numpy())
        errors_pred.extend(preds[mask].cpu().numpy())
        if len(errors_imgs) >= 20:
            break

fig, axes = plt.subplots(4, 5, figsize=(13, 11))
fig.suptitle('DenseNet — Exemples de mauvaises prédictions', fontsize=13, fontweight='bold')

mean, std = 0.2860, 0.3530
for i, ax in enumerate(axes.flat):
    if i >= len(errors_imgs):
        ax.axis('off')
        continue
    img = errors_imgs[i].squeeze().numpy() * std + mean
    ax.imshow(img, cmap='gray')
    ax.set_title(f'✅ {CLASS_NAMES[errors_true[i]]}\n❌ {CLASS_NAMES[errors_pred[i]]}',
                 fontsize=8, color='darkred')
    ax.axis('off')

plt.tight_layout()
plt.savefig('error_examples.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── Accuracy par classe ───────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))

x     = np.arange(len(CLASS_NAMES))
width = 0.35

def per_class_acc(preds, labels):
    cm = confusion_matrix(labels, preds)
    return cm.diagonal() / cm.sum(axis=1) * 100

cnn_class_acc = per_class_acc(cnn_preds, cnn_labels)
dn_class_acc  = per_class_acc(dn_preds,  dn_labels)

bars1 = ax.bar(x - width/2, cnn_class_acc, width, label='CNN',      color='#4C8BF5', alpha=0.85)
bars2 = ax.bar(x + width/2, dn_class_acc,  width, label='DenseNet', color='#00C897', alpha=0.85)

ax.set(xticks=x, xticklabels=CLASS_NAMES, ylabel='Accuracy (%)',
       title='Accuracy par classe — CNN vs DenseNet', ylim=(50, 105))
ax.set_xticklabels(CLASS_NAMES, rotation=30, ha='right')
ax.axhline(y=100, color='gray', linestyle='--', alpha=0.4)
ax.legend()
ax.grid(axis='y', alpha=0.3)
ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('per_class_accuracy.png', dpi=120, bbox_inches='tight')
plt.show()

## 8. 📋 Rapport de classification

In [ ]:
print('\n' + '═' * 60)
print('  📊  Classification Report — CNN')
print('═' * 60)
print(classification_report(cnn_labels, cnn_preds, target_names=CLASS_NAMES, digits=4))

print('\n' + '═' * 60)
print('  🌲  Classification Report — DenseNet')
print('═' * 60)
print(classification_report(dn_labels, dn_preds, target_names=CLASS_NAMES, digits=4))

## 9. 🏆 Comparaison finale & Sauvegarde

In [ ]:
# ── Tableau de comparaison ────────────────────────────────────────────────────
print('\n' + '═' * 62)
print(f'  {"Métrique":<22} {"CNN":>15} {"DenseNet":>15}')
print('─' * 62)
print(f'  {"Accuracy (Test)":<22} {cnn_acc*100:>14.2f}% {dn_acc*100:>14.2f}%')
print(f'  {"Paramètres":<22} {cnn_params:>15,} {dn_params:>15,}')
print(f'  {"Epochs entraînés":<22} {len(cnn_history["train_loss"]):>15} {len(dn_history["train_loss"]):>15}')
print(f'  {"Meilleure val_acc":<22} {max(cnn_history["val_acc"])*100:>14.2f}% {max(dn_history["val_acc"])*100:>14.2f}%')
winner = 'DenseNet' if dn_acc > cnn_acc else 'CNN'
print('═' * 62)
print(f'  🏆  Meilleur modèle : {winner}')
print('═' * 62)

In [ ]:
# ── Sauvegarde des modèles ────────────────────────────────────────────────────
torch.save({
    'model_state_dict' : cnn_model.state_dict(),
    'test_accuracy'    : cnn_acc,
    'config'           : CONFIG,
}, 'cnn_fashionmnist.pth')

torch.save({
    'model_state_dict' : dn_model.state_dict(),
    'test_accuracy'    : dn_acc,
    'config'           : CONFIG,
}, 'densenet_fashionmnist.pth')

print('✅ Modèles sauvegardés :')
print('   - cnn_fashionmnist.pth')
print('   - densenet_fashionmnist.pth')

## ✅ Conclusion

Ce notebook compare deux architectures sur FashionMNIST :

| | CNN | DenseNet |
|---|---|---|
| **Points forts** | Rapide, peu de paramètres | Meilleure réutilisation des features, meilleure généralisation |
| **Points faibles** | Moins expressif | Plus lent à entraîner |
| **Usage recommandé** | Prototypage rapide, ressources limitées | Production, précision maximale |

### Améliorations futures
- Tester **ResNet** ou **EfficientNet** adapté
- Ajouter **Mixup / CutMix** pour une meilleure augmentation
- Utiliser **Optuna** pour l'hyperparameter search automatique
- Déployer avec **ONNX** pour l'inférence en production